In [1]:
# work_item_id = 427761  # e.g., User Story ID
# pr_number = 95  # PR number

# work_item_id = 427761  # e.g., User Story ID
# pr_number = 95  # PR number

# work_item_id = 428346  # e.g., User Story ID
# pr_number = 92  # PR number

# work_item_id = 423483  # e.g., User Story ID
# pr_number = 61  # PR number

# work_item_id = 424599  # e.g., User Story ID
# pr_number = 60  # PR number

# work_item_id = 415267  # e.g., User Story ID
# pr_number = 49  # PR number

work_item_id = 430529
pr_number = 109


# Replace these values
organization = "AdtalemGlobalEducation"
project = "PROD"

# -------- Example Usage --------
owner = "atge-org"
repo = "wu-impacthub-back"

In [2]:
import os
from bs4 import BeautifulSoup
import html
import requests
from requests.auth import HTTPBasicAuth
from dotenv import load_dotenv

load_dotenv()

pat = os.getenv("PAT")
key = os.getenv("KEY")
token = os.getenv("TOKEN")

**-------Work Item - AC Details Fetching--------**

In [3]:
personal_access_token = pat  # Azure DevOps PAT with Work Item read scope

url = f"https://dev.azure.com/AdtalemGlobalEducation/PROD/_apis/wit/workitems/{work_item_id}?fields=Microsoft.VSTS.Common.AcceptanceCriteria&api-version=7.1-preview.3"

# Azure DevOps requires username:PAT as basic auth (username can be empty)
response = requests.get(
    url,
    auth=HTTPBasicAuth("", pat),
    verify=False,
    proxies={"http": None, "https": None},
)

data = response.json()

html_text = data["fields"]["Microsoft.VSTS.Common.AcceptanceCriteria"]

c:\Users\sameer.mohammed\Documents\Aspire\pr_agent_dec\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'dev.azure.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [4]:
def remove_div_tags(html_text):
    if not html_text:
        return ""

    # Decode escaped sequences if present
    decoded = html.unescape(html_text)

    soup = BeautifulSoup(decoded, "html.parser")

    # Remove all <div> tags but keep their text
    for div in soup.find_all("div"):
        div.unwrap()

    # Get clean text
    clean = soup.get_text(separator="\n")

    # Remove duplicate blank lines
    clean = "\n".join(line.strip() for line in clean.split("\n") if line.strip())

    clean = clean.replace("\n", "")

    clean = clean.replace("\xa0", "")

    return clean

In [5]:
acceptance_criteria = remove_div_tags(html_text)

**--------Fetching PR details from GIT---------**

In [6]:
def get_pr_files(owner, repo, pr_number, token):
    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}/files"

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github.v3+json",
    }

    response = requests.get(url, headers=headers)

    # Raise error if something goes wrong
    response.raise_for_status()

    return response.json()

files = get_pr_files(owner, repo, pr_number, token)

for f in files:
    print("Filename:", f["filename"])
    print("Status:", f["status"])
    print("Additions:", f["additions"])
    print("Deletions:", f["deletions"])
    print("Changes:", f["changes"])
    print("Patch Diff:\n", f.get("patch"))
    print("-" * 50)

pr_code =  f.get("patch")

Filename: fn_fih_student_progress/main.py
Status: modified
Additions: 5
Deletions: 4
Changes: 9
Patch Diff:
 @@ -11,7 +11,6 @@
 from firebase_admin import firestore
 from firebase_admin import credentials
 
-
 app = Flask(__name__)
 
 # Set global options for Firebase Functions
@@ -103,7 +102,6 @@ def fn_fih_student_progress(req: https_fn.Request) -> https_fn.Response:
         if not course_end_date:
             return jsonify({"error": "course_end_date is missing in request"}), 400
 
-
         try:
             with engine.connect() as conn:
                 # Query to fetch active notes for the given faculty_id
@@ -139,7 +137,8 @@ def fn_fih_student_progress(req: https_fn.Request) -> https_fn.Response:
                 course_start_date, --28
                 course_end_date, --29
                 no_of_late_assignments, --30
-                no_of_late_discussion_posts --31
+                no_of_late_discussion_posts, --31
+                canvas_sync_date --32
                 

**--------Agent code -- MS Autogen---------**

In [7]:
import asyncio
from pydantic import BaseModel
from typing import List, Literal
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import StructuredMessage
from autogen_agentchat.ui import Console


# ---------------------------------------
# Define Pydantic Model for JSON Response
# ---------------------------------------
class PRValidationResponse(BaseModel):
    validation_result: str  # High-level summary
    match_score: str  # % score

    comments: List[str]  # Findings mapped to AC
    missing_requirements: List[str]  # AC elements not implemented

    code_suggestions: List[str]  # Actionable improvements
    risk_level: Literal["low", "medium", "high"]  # Based on impact of issues

    line_references: List[str]  # e.g., "Line 42: Add null-check"
    potential_bugs: List[str]  # Any risky or buggy patterns detected

    code_quality_score: str  # e.g., “8/10”
    code_smells: List[str]  # e.g., “Long function”, “Magic values”

    priority: Literal["P1", "P2", "P3"]  # Suggested urgency of fixes

    status: Literal["PASS", "FAIL"]  # pass/fail validation


# ---------------------------------------
# Create OpenAI model client
# ---------------------------------------
model_client_openai = OpenAIChatCompletionClient(
    model="gpt-4.1-nano", api_key=key, temperature=0.1
)

# ---------------------------------------
# PR Validation Agent with Structured Output
# ---------------------------------------
agent = AssistantAgent(
    name="PRValidator",
    model_client=model_client_openai,
    system_message="""
        You are a PR Validation Agent.

Your job is to:
1. Compare the Acceptance Criteria (AC) with the submitted PR code.
2. Identify whether the PR code satisfies each part of the AC.
3. Detect missing logic, incomplete implementation, or deviations from expected behavior.
4. Provide specific, actionable code suggestions where improvements are required.
5. Always respond ONLY in valid JSON. No explanations outside JSON.

Your analysis must include:
- Functional correctness based on AC
- Missing conditions or features
- Incorrect or risky logic
- Code quality issues (naming, structure, readability)
- Edge-case handling
- Security, performance, or reliability concerns (if relevant)

You MUST output JSON in the following format:

{
  "validation_result": "High-level summary of how well the PR matches the AC.",
  "match_score": "Percentage match (e.g. 90%) based on completeness.",
  "comments": [
      "Bullet-point analysis describing which AC items are met or missing.",
      "Mention any major issues or logic gaps."
  ],
  "code_suggestions": [
      "Direct, actionable, line-level improvement suggestions.",
      "Recommended refactoring or error handling improvements.",
      "Better variable naming, structure, performance enhancements.",
      "Any missing logic required to fully meet the AC."
  ]
}

RESTRICTIONS:
- Output MUST be strictly valid JSON.
- No markdown, no natural language outside JSON.
- IF unsure, still produce the JSON based on your best inference.

    """,
    description="Validates PR code against Acceptance Criteria.",
    model_client_stream=True,
    output_content_type=PRValidationResponse,  # <-- Enforce structured JSON output
    reflect_on_tool_use=False,
)

# ---------------------------------------
# Build the task
# ---------------------------------------
_task = f"""
Validate the PR code against the acceptance criteria.

Acceptance Criteria:
{acceptance_criteria}

PR Code:
{pr_code}

Return ONLY JSON in the defined format.
"""


# ---------------------------------------
# Run the agent and print only structured output
# ---------------------------------------
async def main():
    result = await Console(agent.run_stream(task=_task))

    json_output = None

    # Extract the final structured message
    last_msg = result.messages[-1]

    if isinstance(last_msg, StructuredMessage):
        content: PRValidationResponse = last_msg.content

        # Convert structured Pydantic model → JSON dictionary
        json_output = content.model_dump()

    await model_client_openai.close()

    return json_output


response_json = await main()

---------- TextMessage (user) ----------

Validate the PR code against the acceptance criteria.

Acceptance Criteria:
In FIH, display the canvas data extraction date-time

PR Code:
@@ -11,7 +11,6 @@
 from firebase_admin import firestore
 from firebase_admin import credentials
 
-
 app = Flask(__name__)
 
 # Set global options for Firebase Functions
@@ -103,7 +102,6 @@ def fn_fih_student_progress(req: https_fn.Request) -> https_fn.Response:
         if not course_end_date:
             return jsonify({"error": "course_end_date is missing in request"}), 400
 
-
         try:
             with engine.connect() as conn:
                 # Query to fetch active notes for the given faculty_id
@@ -139,7 +137,8 @@ def fn_fih_student_progress(req: https_fn.Request) -> https_fn.Response:
                 course_start_date, --28
                 course_end_date, --29
                 no_of_late_assignments, --30
-                no_of_late_discussion_posts --31
+                no_of_late_discuss

**--------SMART PR AI Validation Metrics-------**

In [8]:
print("**--------SMART PR AI Validation Metrics-------**")
print("--------------------------------------------------")

print("status: ",response_json["status"])
print("")
print("match_score:",response_json["match_score"])
print("")
print("validation_result: ",response_json["validation_result"])
print("")
print("comments:",response_json["comments"])
print("")
print("code_suggestions:",response_json["code_suggestions"])
print("")
print("missing_requirements:",response_json["missing_requirements"])
print("")
print("risk_level:",response_json["risk_level"])
print("")
print("potential_bugs:",response_json["potential_bugs"])
print("")
print("code_quality_score:",response_json["code_quality_score"])
print("")
print("code_smells:",response_json["code_smells"])
print("")
print("priority:",response_json["priority"])

**--------SMART PR AI Validation Metrics-------**
--------------------------------------------------
status:  PASS

match_score: 80%

validation_result:  The PR code has been modified to include the display of the canvas data extraction date-time, aligning with the acceptance criteria. The relevant data is fetched from the database and included in the response.

comments: ["The code now extracts 'canvas_sync_date' from the database and includes it in the response under 'studentProgressExtractionTimestamp', fulfilling the AC requirement.", 'The code correctly retrieves and displays the canvas data extraction date-time in the student progress data.', "The code snippet shows proper integration of the new data point, but it lacks explicit validation or handling for the presence of 'canvas_sync_date' in the database rows.", 'The code could benefit from additional comments or documentation to clarify the purpose of the new field.']

code_suggestions: ["Add validation to check if 'canvas_sync